In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

import random
import statistics
import numpy as np

from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split

SEED = 42

In [2]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

Device: cuda


In [ ]:
df = load_wine()
print(df)

In [ ]:
X = df.data
y = df.target

In [ ]:
# X = np.array([[random.random() for col in range(13)] for row in range(1_000_000)])
# y = np.array([random.randint(0, 2) for i in range(1_000_000)])

In [ ]:
# Scaling (standard transformation).
for var in range(X.shape[1]):
    mean = statistics.mean(X[:, var])
    std = statistics.stdev(X[:, var])
    X[:, var] = (X[:, var] - mean) / std

In [7]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)

In [8]:
# We use FloatTensor() for X because it keeps the inputs as 32bit floats.
# They are small enough to be fast but with enough detail. In this dataset all the
# inputs are floats. 32bits is also optimizad for cuda libraries.
# For y we use LongTensor() because the output in this dataset can be 0, 1, 2.
# LongTensor() converts to 64bit integers and works well in this case.
X_train = torch.FloatTensor(X_train)
y_train = torch.LongTensor(y_train)

In [9]:
train_ds = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_ds, batch_size=2048, shuffle=True)

In [10]:
class WineNet(nn.Module):
    def __init__(self):
        super(WineNet, self).__init__()
        
        # First layes needs 13 inputs (dataset has 13 features).
        self.layer1 = nn.Linear(13, 32)
        self.layer2 = nn.Linear(32, 16)
        self.output = nn.Linear(16, 3)
        self.relu = nn.ReLU()
        
    def forward(self, x):
        x = self.layer1(x)
        x = self.relu(x)
        x = self.layer2(x)
        x = self.relu(x)
        x = self.output(x)
        return x

In [11]:
model = WineNet().to(DEVICE)

In [12]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

for epoch in range(15):
    total_loss = 0
    total_correct = 0
    total_samples = 0
    
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        _, pred = torch.max(outputs, 1)
        total_samples += labels.size(0)
        total_correct += (pred == labels).sum().item()
    
    acc = total_correct / total_samples
    print(f'Epoch: {epoch + 1}, accuracy: {acc}, loss: {total_loss}')

Epoch: 1, accuracy: 0.33406125, loss: 429.7429995536804
Epoch: 2, accuracy: 0.3340825, loss: 429.5801992416382
Epoch: 3, accuracy: 0.3349, loss: 429.56806099414825
Epoch: 4, accuracy: 0.334275, loss: 429.57286047935486
Epoch: 5, accuracy: 0.33451625, loss: 429.561194896698
Epoch: 6, accuracy: 0.33470875, loss: 429.5585708618164
Epoch: 7, accuracy: 0.33467375, loss: 429.5599310398102
Epoch: 8, accuracy: 0.33567375, loss: 429.5522212982178
Epoch: 9, accuracy: 0.335155, loss: 429.55130422115326
Epoch: 10, accuracy: 0.3354975, loss: 429.5496771335602
Epoch: 11, accuracy: 0.335505, loss: 429.5502495765686
Epoch: 12, accuracy: 0.33624125, loss: 429.5458515882492
Epoch: 13, accuracy: 0.33563625, loss: 429.54568886756897
Epoch: 14, accuracy: 0.33638375, loss: 429.54290080070496
Epoch: 15, accuracy: 0.33638375, loss: 429.53914391994476
